
# Rectified Flow 论文精读笔记

**论文：** *Flow Straight and Fast: Learning to Generate and Transfer Data with Rectified Flow*  
**作者：** Xingchao Liu, Chengyue Gong, Qiang Liu  
**会议：** ICLR 2023

---

# 第一部分：论文到底在解决什么问题？

Rectified Flow 要解决的是一个概率分布传输问题。

给定两个分布：

$$
X_0\sim \pi_0,\qquad X_1\sim\pi_1,
\tag{1.1}
$$

其中在生成任务中通常：

$$
\pi_0=\mathcal N(0,I),
\qquad
\pi_1=p_{\mathrm{data}}.
\tag{1.2}
$$

我们希望找到一个确定性映射

$$
T:\mathbb R^d\rightarrow\mathbb R^d,
\tag{1.3}
$$

使得：

$$
Z_0\sim\pi_0
\quad\Longrightarrow\quad
Z_1=T(Z_0)\sim\pi_1.
\tag{1.4}
$$

也就是说，把简单分布 $\pi_0$ 输运到复杂分布 $\pi_1$。

Rectified Flow 不直接学习 $T$，而是学习 ODE：

$$
\boxed{
\frac{dZ_t}{dt}=v(Z_t,t)
}
\tag{1.5}
$$

从 $t=0$ 积分到 $t=1$：

$$
Z_1
=
Z_0+\int_0^1v(Z_t,t)\,dt.
\tag{1.6}
$$

问题在于，如果 $v(Z_t,t)$ 很复杂、轨迹很弯，就需要很多 ODE 积分步。

论文因此希望得到一种特殊的 ODE：

$$
Z_t=(1-t)Z_0+tZ_1,
\tag{1.7}
$$

也就是**匀速直线**。

此时：

$$
\dot Z_t=Z_1-Z_0=\mathrm{const},
\tag{1.8}
$$

从而：

$$
Z_1=Z_0+v(Z_0,0).
\tag{1.9}
$$

理论上一个 Euler step 就够。

因此整篇论文的核心问题是：

$$
\boxed{
\text{如何在保持目标分布正确的同时，让生成 ODE 尽可能接近匀速直线？}
}
$$

论文提出两个关键过程：

$$
\boxed{
\text{Rectification：重新组织 coupling}
}
$$

以及

$$
\boxed{
\text{Reflow：反复 Rectification，使 flow 越来越直。}
}
$$

原论文 Figure 2 就把这个过程描述为：原始直线 interpolation 可以交叉，Rectified Flow 在交叉处进行 rewiring，同时保持每个时间的边缘分布。

---

# 第二部分：理解论文前必须掌握的数学基础

# 2.1 随机变量和概率分布

记：

$$
X\sim p_X.
\tag{2.1}
$$

这里：

- $X$ 是随机变量；
- $p_X$ 是它的概率分布；
- $x$ 是某一次具体采样值。

Rectified Flow 中经常出现：

$$
X_0,\quad X_1,\quad X_t,\quad U,\quad Z_t.
\tag{2.2}
$$

这些都是随机变量。

而：

$$
\pi_0,\quad\pi_1,\quad p_t
\tag{2.3}
$$

则表示分布。

---

# 2.2 联合分布与耦合

仅仅知道：

$$
X_0\sim\pi_0,\qquad X_1\sim\pi_1
\tag{2.4}
$$

并不能确定 $X_0$ 和 $X_1$ 怎么配对。

例如：

$$
X_0\sim\mathcal N(0,1),\qquad
X_1\sim\mathcal N(0,4).
$$

既可以有：

$$
X_1=2X_0,
$$

也可以：

$$
X_1=-2X_0,
$$

还可以让：

$$
X_0\perp X_1.
$$

它们具有相同的两个边缘分布，但联合关系完全不同。

因此定义：

$$
\boxed{
(X_0,X_1)\sim\gamma.
}
\tag{2.5}
$$

如果 $\gamma$ 的两个边缘分别为 $\pi_0,\pi_1$，则称：

$$
\boxed{
\gamma\in\Pi(\pi_0,\pi_1)
}
\tag{2.6}
$$

是 $\pi_0,\pi_1$ 的一个 **coupling（耦合）**。

所以最简单地理解：

$$
\boxed{
\text{coupling = “哪个 }X_0\text{ 和哪个 }X_1\text{ 配成一对”的概率规则。}
}
$$

你的原笔记也是从这个角度定义 coupling 的。

生成模型第一轮训练常用的是独立耦合：

$$
\boxed{
\gamma_0=\pi_0\otimes\pi_1.
}
\tag{2.7}
$$

即：

$$
X_0\perp X_1.
$$

实际就是：

$$
x_0\sim\mathcal N(0,I),
\qquad
x_1\sim p_{\mathrm{data}}
$$

分别独立采样。

---

# 2.3 条件分布和条件期望

考虑两个随机变量：

$$
A,\qquad B.
$$

如果已知：

$$
A=a,
$$

那么 $B$ 仍然可能有多个取值，因此存在：

$$
p(B\mid A=a).
\tag{2.8}
$$

其条件期望定义为：

$$
\boxed{
\mathbb E[B\mid A=a]
=
\int b\,p(b\mid a)\,db.
}
\tag{2.9}
$$

这个概念是 Rectified Flow 最重要的数学基础。

---

# 2.4 为什么 MSE 会学到条件期望？

假设我们用函数 $f(A)$ 预测 $B$：

$$
\mathcal L(f)
=
\mathbb E\|f(A)-B\|^2.
\tag{2.10}
$$

固定：

$$
A=a.
$$

问题变成：

$$
\mathcal L_a
=
\mathbb E[
\|f(a)-B\|^2
\mid A=a].
\tag{2.11}
$$

展开：

$$
\begin{aligned}
\mathcal L_a
&=
\|f(a)\|^2
-2f(a)^T\mathbb E[B\mid A=a]
\\
&\quad+
\mathbb E[\|B\|^2\mid A=a].
\end{aligned}
\tag{2.12}
$$

对 $f(a)$ 求导：

$$
\frac{\partial\mathcal L_a}{\partial f(a)}
=
2f(a)
-
2\mathbb E[B\mid A=a].
\tag{2.13}
$$

令其为零：

$$
\boxed{
f^*(a)=\mathbb E[B\mid A=a].
}
\tag{2.14}
$$

因此：

$$
\boxed{
\arg\min_f
\mathbb E\|f(A)-B\|^2
=
\mathbb E[B\mid A].
}
\tag{2.15}
$$

这就是后面边际速度出现的原因。

---

# 2.5 Jensen 不等式

若：

$$
c:\mathbb R^d\rightarrow\mathbb R
$$

是凸函数，则：

$$
\boxed{
c(\mathbb E[X])
\leq
\mathbb E[c(X)].
}
\tag{2.16}
$$

条件版本：

$$
\boxed{
c(\mathbb E[X\mid Y])
\leq
\mathbb E[c(X)\mid Y].
}
\tag{2.17}
$$

时间积分也可以看成均匀分布下的期望。

因为区间长度是 1：

$$
\boxed{
c\left(
\int_0^1 f(t)dt
\right)
\leq
\int_0^1c(f(t))dt.
}
\tag{2.18}
$$

后面证明 convex transport cost 不增时会连续使用两次 Jensen。

---

# 2.6 ODE 与连续性方程

考虑 ODE：

$$
\dot Z_t=v(Z_t,t).
\tag{2.19}
$$

它描述单个粒子的运动。

如果大量粒子都按照这个速度场运动，记它们的概率密度为 $p_t(x)$，那么：

$$
\boxed{
\partial_t p_t(x)
+
\nabla\cdot[p_t(x)v(x,t)]
=
0.
}
\tag{2.20}
$$

这叫连续性方程。

物理意义就是：

$$
\boxed{
\text{概率质量守恒。}
}
$$

它是后面证明：

$$
\operatorname{Law}(Z_t)
=
\operatorname{Law}(X_t)
$$

的核心。

---

# 第三部分：从任意 coupling 构造 Rectified Flow

现在正式进入论文。

给定：

$$
\boxed{
(X_0,X_1)\sim\gamma_0,
\qquad
\gamma_0\in\Pi(\pi_0,\pi_1).
}
\tag{3.1}
$$

这里的 coupling 可以是独立 coupling，也可以是任意已有 pairing。

---

# 3.1 构造线性 interpolation

定义：

$$
\boxed{
X_t=(1-t)X_0+tX_1.
}
\tag{3.2}
$$

如果固定一次采样：

$$
X_0=x_0,\qquad X_1=x_1,
$$

则：

$$
x_t=(1-t)x_0+tx_1.
$$

这是一条直线。

对式 (3.2) 求导：

$$
\dot X_t
=
-X_0+X_1.
$$

因此：

$$
\boxed{
\dot X_t=X_1-X_0.
}
\tag{3.3}
$$

为了方便，定义：

$$
\boxed{
U=X_1-X_0.
}
\tag{3.4}
$$

所以：

$$
\dot X_t=U.
\tag{3.5}
$$

这里的 $U$ 是随机变量，因为 $(X_0,X_1)$ 是随机的。

但是对于一对固定端点：

$$
(x_0,x_1),
$$

对应：

$$
u=x_1-x_0
$$

是恒定速度。

我们可以把它理解成：

$$
\boxed{
\text{pairwise / conditional velocity}.
}
$$

你的原笔记也是从任意 coupling 先构造这条线性路径。

---

# 3.2 为什么不能直接用 $U$ 做生成？

因为：

$$
U=X_1-X_0.
$$

它需要知道终点 $X_1$。

训练时我们同时有：

$$
X_0,\quad X_1.
$$

但是生成时只有：

$$
Z_0\sim\pi_0.
$$

未来的 $Z_1$ 是不知道的。

因此：

$$
\dot X_t=X_1-X_0
$$

虽然是一条漂亮的直线，却是一个 **anticipating / non-causal process**。

论文因此希望构造：

$$
\boxed{
v(x,t)
}
$$

即：

> 只知道当前位置 $x$ 和时间 $t$，就可以决定速度。

原论文明确称这个过程为对 linear interpolation 的 “causalization”。

---

# 3.3 为什么同一个 $X_t$ 可能有多个速度？

例如一维：

第一条：

$$
X_0=-1,\qquad X_1=1,
$$

那么在：

$$
t=\frac12
$$

时：

$$
X_t=0,
\qquad
U=2.
$$

第二条：

$$
X_0=1,\qquad X_1=-1,
$$

同样：

$$
X_t=0,
$$

但是：

$$
U=-2.
$$

因此：

$$
X_t=0
$$

并不能唯一决定 $U$。

所以存在：

$$
\boxed{
p(U\mid X_t=x).
}
\tag{3.6}
$$

这就是为什么不能令：

$$
v(x,t)=U.
$$

---

# 第四部分：从条件速度推导边际速度场

Rectified Flow 的训练目标是：

$$
\boxed{
\mathcal L(v)
=
\int_0^1
\mathbb E_{\gamma_0}
\left[
\|v(X_t,t)-(X_1-X_0)\|^2
\right]dt.
}
\tag{4.1}
$$

也可以写成：

$$
\boxed{
\mathcal L(v)
=
\mathbb E_{
\substack{
t\sim U(0,1)\\
(X_0,X_1)\sim\gamma_0
}}
\|v(X_t,t)-U\|^2.
}
\tag{4.2}
$$

根据前面 MSE 的结论：

$$
f^*(A)=\mathbb E[B\mid A],
$$

现在做变量对应：

$$
A\longleftrightarrow (X_t,t),
$$

$$
B\longleftrightarrow U.
$$

直接得到：

$$
\boxed{
v^*(x,t)
=
\mathbb E[U\mid X_t=x].
}
\tag{4.3}
$$

再代入：

$$
U=X_1-X_0,
$$

得到：

$$
\boxed{
v^*(x,t)
=
\mathbb E[X_1-X_0\mid X_t=x].
}
\tag{4.4}
$$

这就是论文 Equation (2)。原论文将其解释为：对所有在时间 $t$ 经过 $x$ 的直线方向 $X_1-X_0$ 取条件平均。

这就是所谓 **marginal velocity field**。

---

# 4.1 为什么叫边际速度？

假设：

$$
(X_t,U)\sim\rho_t.
\tag{4.5}
$$

那么：

$$
\rho_t(u\mid x)
$$

表示：

> 已知在时刻 $t$ 到达了位置 $x$，原来的 endpoint pair 可能对应哪些速度 $u$。

于是：

$$
\boxed{
v^*(x,t)
=
\int u\,\rho_t(du\mid x).
}
\tag{4.6}
$$

也就是说：

$$
\boxed{
\text{把 endpoint identity 积分掉，只保留当前位置 }x,
}
$$

所以称为 marginal velocity。

---

# 4.2 实际训练并不需要知道这个条件分布

理论上存在：

$$
p(U\mid X_t=x),
$$

但实际往往无法解析计算。

训练只需要：

$$
x_0\sim\pi_0,
$$

$$
x_1\sim\pi_1,
$$

$$
t\sim U(0,1),
$$

然后构造：

$$
x_t=(1-t)x_0+tx_1,
\tag{4.7}
$$

$$
u=x_1-x_0.
\tag{4.8}
$$

直接优化：

$$
\|v_\theta(x_t,t)-u\|^2.
\tag{4.9}
$$

在总体最优意义下，MSE 自动学到：

$$
v_\theta(x,t)\rightarrow
\mathbb E[U\mid X_t=x].
$$

所以：

$$
\boxed{
\text{我们不计算条件期望，而是通过 MC + MSE 回归它。}
}
$$

---

# 4.3 原论文 Equation (4) 怎么理解？

论文还进一步假设：

$$
X_0\mid X_1=x_1
$$

存在条件密度：

$$
\rho(x_0\mid x_1).
$$

已知：

$$
X_t=z
$$

和：

$$
X_1=x_1.
$$

从：

$$
z=(1-t)X_0+tX_1
$$

反解：

$$
\boxed{
X_0
=
\frac{z-tX_1}{1-t}.
}
\tag{4.10}
$$

于是：

$$
X_1-X_0
=
X_1-\frac{z-tX_1}{1-t}.
$$

通分：

$$
=
\frac{(1-t)X_1-z+tX_1}{1-t}.
$$

整理：

$$
\boxed{
X_1-X_0
=
\frac{X_1-z}{1-t}.
}
\tag{4.11}
$$

因此：

$$
v^X(z,t)
=
\mathbb E
\left[
\frac{X_1-z}{1-t}
\eta_t(X_1,z)
\right],
\tag{4.12}
$$

其中 $\eta_t$ 本质上就是在条件 $X_t=z$ 下不同 $X_1$ 的 posterior 权重。论文给出了它由 $\rho(X_0\mid X_1)$ 表达的形式。

这一部分只是对：

$$
\mathbb E[X_1-X_0\mid X_t=z]
$$

进行显式展开。

实际神经网络训练并不需要计算它。

---

# 第五部分：为什么这个边际速度真的能正确运输分布？

现在出现两个完全不同的过程。

训练使用的插值过程：

$$
\boxed{
X_t=(1-t)X_0+tX_1.
}
\tag{5.1}
$$

真正生成使用的是：

$$
\boxed{
\dot Z_t=v^*(Z_t,t),
\qquad
Z_0\sim\pi_0.
}
\tag{5.2}
$$

单个粒子一般：

$$
Z_t\neq X_t.
\tag{5.3}
$$

论文 Theorem 3.3 要证明的是：

$$
\boxed{
\operatorname{Law}(Z_t)
=
\operatorname{Law}(X_t),
\qquad
\forall t\in[0,1].
}
\tag{5.4}
$$

原论文把它称为 marginal preserving property。

---

# 5.1 第一步：求 $X_t$ 的概率演化

记：

$$
p_t=\operatorname{Law}(X_t).
\tag{5.5}
$$

取任意光滑测试函数：

$$
h:\mathbb R^d\rightarrow\mathbb R.
$$

考虑：

$$
F(t)=\mathbb E[h(X_t)].
\tag{5.6}
$$

对时间求导：

$$
\frac{dF}{dt}
=
\mathbb E
\left[
\frac{d}{dt}h(X_t)
\right].
\tag{5.7}
$$

链式法则：

$$
\frac{d}{dt}h(X_t)
=
\nabla h(X_t)^T\dot X_t.
$$

代入：

$$
\frac{dF}{dt}
=
\mathbb E[
\nabla h(X_t)^T\dot X_t].
\tag{5.8}
$$

因为：

$$
\dot X_t=U,
$$

所以：

$$
=
\mathbb E[
\nabla h(X_t)^TU].
\tag{5.9}
$$

使用全期望公式：

$$
=
\mathbb E
\left[
\mathbb E[
\nabla h(X_t)^TU
\mid X_t]
\right].
\tag{5.10}
$$

给定 $X_t$ 后：

$$
\nabla h(X_t)
$$

已经确定，可以提出来：

$$
=
\mathbb E
\left[
\nabla h(X_t)^T
\mathbb E[U\mid X_t]
\right].
\tag{5.11}
$$

根据边际速度定义：

$$
\mathbb E[U\mid X_t]
=
v^*(X_t,t).
$$

所以：

$$
\boxed{
\frac d{dt}
\mathbb E[h(X_t)]
=
\mathbb E[
\nabla h(X_t)^Tv^*(X_t,t)].
}
\tag{5.12}
$$

将期望写成积分：

$$
\frac d{dt}
\int h(x)p_t(x)dx
=
\int
\nabla h(x)^Tv^*(x,t)p_t(x)dx.
\tag{5.13}
$$

对右边做分部积分：

$$
=
-\int
h(x)\nabla\cdot[p_t(x)v^*(x,t)]dx.
\tag{5.14}
$$

所以：

$$
\boxed{
\partial_t p_t+
\nabla\cdot(p_tv^*)=0.
}
\tag{5.15}
$$

这正是连续性方程。你的原笔记也按这条弱形式推导展开了 Theorem 3.3。

---

# 5.2 第二步：ODE 的分布也满足相同方程

对于：

$$
\dot Z_t=v^*(Z_t,t),
$$

记：

$$
q_t=\operatorname{Law}(Z_t).
$$

ODE 的概率分布满足：

$$
\boxed{
\partial_tq_t+
\nabla\cdot(q_tv^*)=0.
}
\tag{5.16}
$$

而初始条件：

$$
p_0=\pi_0,
\qquad
q_0=\pi_0.
\tag{5.17}
$$

所以：

$$
p_0=q_0.
$$

在适当正则性和唯一性条件下：

$$
\boxed{
p_t=q_t.
}
\tag{5.18}
$$

即：

$$
\boxed{
X_t\overset d=Z_t.
}
\tag{5.19}
$$

但再次强调：

$$
\boxed{
X_t\overset d=Z_t
\quad\neq\quad
X_t=Z_t.
}
$$

它们保持的是**边缘分布**，不是单个粒子的身份。

---

# 第六部分：Rectification 到底改变了什么？

原来的 coupling：

$$
\gamma_{\mathrm{old}}
=
\operatorname{Law}(X_0,X_1).
\tag{6.1}
$$

训练完以后求解 ODE：

$$
\dot Z_t=v^*(Z_t,t).
$$

如果 ODE 解唯一，那么：

$$
Z_1=T(Z_0).
\tag{6.2}
$$

因此产生新的 coupling：

$$
\boxed{
\gamma_{\mathrm{new}}
=
\operatorname{Law}(Z_0,Z_1).
}
\tag{6.3}
$$

也可以写成：

$$
\gamma_{\mathrm{new}}
=
(\mathrm{Id},T)_\#\pi_0.
\tag{6.4}
$$

因为：

$$
Z_0\sim\pi_0,
\qquad
Z_1\sim\pi_1,
$$

所以：

$$
\gamma_{\mathrm{new}}
\in
\Pi(\pi_0,\pi_1).
\tag{6.5}
$$

但是一般：

$$
\boxed{
\gamma_{\mathrm{new}}
\neq
\gamma_{\mathrm{old}}.
}
\tag{6.6}
$$

即：

> 两个边缘分布完全相同，但是 noise-data pairing 被重新组织了。

这就是 Rectification。

---

# 第七部分：为什么凸输运代价一定不增加？

这是论文 Theorem 3.5。

给定任意凸函数：

$$
c:\mathbb R^d\rightarrow\mathbb R.
$$

旧 coupling 的 transport cost：

$$
\boxed{
C_{\mathrm{old}}
=
\mathbb E[c(X_1-X_0)].
}
\tag{7.1}
$$

新 coupling：

$$
\boxed{
C_{\mathrm{new}}
=
\mathbb E[c(Z_1-Z_0)].
}
\tag{7.2}
$$

论文证明：

$$
\boxed{
C_{\mathrm{new}}
\leq
C_{\mathrm{old}}.
}
\tag{7.3}
$$

而且对于所有凸 $c$ 同时成立。

整个证明只有两个核心操作：**两次 Jensen**。

---

# 7.1 第一次 Jensen：沿 ODE 时间方向

根据 ODE：

$$
\dot Z_t=v^*(Z_t,t).
$$

积分：

$$
Z_1-Z_0
=
\int_0^1v^*(Z_t,t)dt.
\tag{7.4}
$$

代入新 cost：

$$
C_{\mathrm{new}}
=
\mathbb E
\left[
c\left(
\int_0^1v^*(Z_t,t)dt
\right)
\right].
\tag{7.5}
$$

使用 Jensen：

$$
c\left(
\int_0^1v_tdt
\right)
\leq
\int_0^1c(v_t)dt.
$$

令：

$$
v_t=v^*(Z_t,t),
$$

于是：

$$
C_{\mathrm{new}}
\leq
\int_0^1
\mathbb E[
c(v^*(Z_t,t))]
dt.
\tag{7.6}
$$

这里要注意：

$$
\boxed{
Z_1-Z_0=\int vdt
}
$$

是 ODE 给出的；

而：

$$
\boxed{
c\left(\int vdt\right)
\leq
\int c(v)dt
}
$$

是凸性/Jensen 给出的。

不是 ODE 自己保证 cost 降低。

---

# 7.2 利用边缘分布保持

前面已经证明：

$$
Z_t\overset d=X_t.
$$

因此：

$$
\mathbb E[
c(v^*(Z_t,t))]
=
\mathbb E[
c(v^*(X_t,t))].
\tag{7.7}
$$

代入式 (7.6)：

$$
C_{\mathrm{new}}
\leq
\int_0^1
\mathbb E[
c(v^*(X_t,t))]
dt.
\tag{7.8}
$$

---

# 7.3 第二次 Jensen：条件速度到边际速度

现在：

$$
v^*(X_t,t)
=
\mathbb E[U\mid X_t].
$$

所以：

$$
c(v^*)
=
c(
\mathbb E[U\mid X_t]).
$$

根据条件 Jensen：

$$
c(
\mathbb E[U\mid X_t])
\leq
\mathbb E[c(U)\mid X_t].
\tag{7.9}
$$

两边对 $X_t$ 求期望：

$$
\mathbb E[
c(v^*(X_t,t))]
\leq
\mathbb E[
\mathbb E[c(U)\mid X_t]].
\tag{7.10}
$$

利用全期望公式：

$$
=
\mathbb E[c(U)].
$$

再代入：

$$
U=X_1-X_0,
$$

得到：

$$
\boxed{
\mathbb E[
c(v^*(X_t,t))]
\leq
\mathbb E[
c(X_1-X_0)].
}
\tag{7.11}
$$

---

# 7.4 合并

由：

$$
C_{\mathrm{new}}
\leq
\int_0^1
\mathbb E[c(v^*(X_t,t))]dt
$$

以及：

$$
\mathbb E[c(v^*(X_t,t))]
\leq C_{\mathrm{old}},
$$

得到：

$$
C_{\mathrm{new}}
\leq
\int_0^1C_{\mathrm{old}}dt.
$$

区间长度为 1：

$$
\boxed{
C_{\mathrm{new}}
\leq
C_{\mathrm{old}}.
}
\tag{7.12}
$$

---

# 7.5 两次 Jensen 分别意味着什么？

第一层：

$$
\boxed{
U
\longrightarrow
\mathbb E[U\mid X_t].
}
$$

把在同一个时空位置出现的多个冲突速度做平均。

所以消除的是：

$$
\boxed{
\text{velocity ambiguity}.
}
$$

第二层：

$$
\boxed{
v(Z_t,t)
\longrightarrow
\int_0^1v(Z_t,t)dt
=
Z_1-Z_0.
}
$$

把沿一条 ODE trajectory 不同时间的速度变化压成最终净位移。

所以对应：

$$
\boxed{
\text{trajectory velocity variation}.
}
$$

这两层正好为后面的 Reflow 直线化理论做准备。

---

# 第八部分：什么叫“直”？

论文中的 straight 比“几何上是一条直线”更严格。

定义：

$$
\boxed{
Z_t=(1-t)Z_0+tZ_1.
}
\tag{8.1}
$$

对时间求导：

$$
\boxed{
\dot Z_t=Z_1-Z_0=\mathrm{const}.
}
\tag{8.2}
$$

所以论文所谓 straight flow 是：

$$
\boxed{
\text{匀速直线 flow}.
}
$$

原论文明确说明，perfectly straight flow 可以用单个 Euler step 精确求终点。

---

# 8.1 直线性指标 $S(Z)$

论文 Equation (3) 定义：

$$
\boxed{
S(\mathbf Z)
=
\int_0^1
\mathbb E
\left[
\|
(Z_1-Z_0)-\dot Z_t
\|^2
\right]dt.
}
\tag{8.3}
$$

由于：

$$
Z_1-Z_0
=
\int_0^1\dot Z_tdt,
\tag{8.4}
$$

它就是整条路径的平均速度。

因此 $S$ 衡量：

$$
\boxed{
\text{瞬时速度 }\dot Z_t
\text{ 与整条轨迹平均速度的偏差。}
}
$$

如果：

$$
S(\mathbf Z)=0,
$$

则几乎处处：

$$
\dot Z_t=Z_1-Z_0.
$$

积分：

$$
Z_t
=
Z_0+t(Z_1-Z_0).
$$

因此：

$$
\boxed{
S(\mathbf Z)=0
\Longleftrightarrow
\mathbf Z\text{ 是匀速直线。}
}
\tag{8.5}
$$

原论文正是用这个量定义 straightness。

---

# 8.2 为什么恒定条件速度 target 不保证 ODE 本身直？

训练 target：

$$
U=X_1-X_0
$$

对于一条训练 interpolation 确实是常数。

但模型学到的是：

$$
v(x,t)
=
\mathbb E[U\mid X_t=x].
$$

粒子真正运动时：

$$
\dot Z_t=v(Z_t,t).
$$

随着 $Z_t$ 移动到不同的：

$$
(Z_t,t),
$$

条件分布：

$$
p(U\mid X_t=Z_t)
$$

也在变化。

所以：

$$
v(Z_t,t)
$$

可能不断变化。

因此第一轮 RF 的 ODE 仍可能是弯的。

---

# 8.3 Burgers 方程与直线性

若：

$$
\dot Z_t=v(Z_t,t)
$$

是一条匀速直线，则沿轨迹：

$$
\frac d{dt}v(Z_t,t)=0.
\tag{8.6}
$$

链式法则：

$$
\frac d{dt}v(Z_t,t)
=
\partial_t v
+
(\nabla v)\dot Z_t.
\tag{8.7}
$$

代入：

$$
\dot Z_t=v,
$$

得到：

$$
\boxed{
\partial_tv+(v\cdot\nabla)v=0.
}
\tag{8.8}
$$

这就是无黏性 Burgers 方程。

所以 Rectified Flow 想要的 straightness，从速度场角度理解就是：

$$
\boxed{
\text{速度沿它自己生成的特征轨迹保持不变。}
}
$$

---

# 第九部分：什么叫 straight coupling？

论文还区分：

$$
\text{straight flow}
$$

和：

$$
\text{straight coupling}.
$$

对于 coupling：

$$
(X_0,X_1)
$$

定义：

$$
X_t=(1-t)X_0+tX_1.
$$

定义速度歧义：

$$
\boxed{
V(X_0,X_1)
=
\int_0^1
\mathbb E
\left[
\|
X_1-X_0
-
\mathbb E[X_1-X_0\mid X_t]
\|^2
\right]dt.
}
\tag{9.1}
$$

如果：

$$
V=0,
$$

则：

$$
X_1-X_0
=
\mathbb E[X_1-X_0\mid X_t].
\tag{9.2}
$$

意味着给定：

$$
X_t=x
$$

以后，速度已经唯一，没有不同方向的直线经过同一个时空位置。

所以：

$$
\boxed{
V=0
\Longleftrightarrow
\text{linear interpolation 不存在 velocity conflict / crossing。}
}
$$

论文 Theorem 3.6 进一步证明，在相应条件下以下性质等价：coupling 是 Rectify 的 fixed point、Rectified Flow 与 linear interpolation 相同、$V=0$，以及严格凸 cost 在 Rectification 后不再下降。

---

# 第十部分：Reflow 到底怎么做？

第一轮：

$$
(X_0,X_1)\sim\gamma_0.
$$

训练：

$$
v_0(x,t)
=
\mathbb E_{\gamma_0}
[X_1-X_0\mid X_t=x].
\tag{10.1}
$$

求解：

$$
\dot Z_t^{(1)}
=
v_0(Z_t^{(1)},t).
\tag{10.2}
$$

得到：

$$
\boxed{
\gamma_1
=
\operatorname{Law}
(Z_0^{(1)},Z_1^{(1)}).
}
\tag{10.3}
$$

Reflow 的关键不是继续沿第一轮 ODE trajectory 训练。

而是：

$$
\boxed{
\text{只保存第一轮 ODE 的两个端点。}
}
$$

采样：

$$
(A_1,B_1)\sim\gamma_1.
$$

重新构造直线：

$$
\boxed{
X_t^{(1)}
=
(1-t)A_1+tB_1.
}
\tag{10.4}
$$

重新定义 target：

$$
\boxed{
U_1=B_1-A_1.
}
\tag{10.5}
$$

再训练：

$$
\boxed{
v_1(x,t)
=
\mathbb E[
U_1\mid X_t^{(1)}=x].
}
\tag{10.6}
$$

再求解 ODE：

$$
\dot Z_t^{(2)}
=
v_1(Z_t^{(2)},t).
\tag{10.7}
$$

得到：

$$
\gamma_2.
$$

因此：

$$
\boxed{
\gamma_0
\rightarrow
\gamma_1
\rightarrow
\gamma_2
\rightarrow
\cdots
}
\tag{10.8}
$$

每一步都是：

$$
\boxed{
\gamma_k
\overset{\text{linear interpolation + regression}}
{\longrightarrow}
v_k
\overset{\text{ODE}}
{\longrightarrow}
\gamma_{k+1}.
}
\tag{10.9}
$$

原论文 Algorithm 1 正是：

$$
\mathbf Z^{k+1}
=
\mathtt{RectFlow}
((Z_0^k,Z_1^k)).
$$


你的原笔记也特别强调了：第二轮不是直接学习第一轮 ODE 的弯曲路径，而是取它的两个端点重新拉直线。

---

# 第十一部分：为什么 Reflow 会越来越直？

这是整篇论文后半部分最重要的理论。

统一第 $k$ 轮符号。

设输入 coupling：

$$
(A_k,B_k)\sim\gamma_k.
\tag{11.1}
$$

定义：

$$
X_t^{(k)}
=
(1-t)A_k+tB_k,
\tag{11.2}
$$

$$
U_k=B_k-A_k.
\tag{11.3}
$$

边际速度：

$$
\boxed{
v_k(x,t)
=
\mathbb E[
U_k\mid X_t^{(k)}=x].
}
\tag{11.4}
$$

ODE：

$$
\dot Z_t^{(k+1)}
=
v_k(Z_t^{(k+1)},t).
\tag{11.5}
$$

下一轮 coupling：

$$
\gamma_{k+1}
=
\operatorname{Law}
(Z_0^{(k+1)},Z_1^{(k+1)}).
\tag{11.6}
$$

---

# 11.1 定义平方输运代价

定义：

$$
\boxed{
C_k
=
\mathbb E\|B_k-A_k\|^2
=
\mathbb E\|U_k\|^2.
}
\tag{11.7}
$$

---

# 11.2 定义 velocity ambiguity

定义：

$$
\boxed{
V_k
=
\int_0^1
\mathbb E
\|
U_k-v_k(X_t^{(k)},t)
\|^2dt.
}
\tag{11.8}
$$

因为：

$$
v_k(X_t,t)
=
\mathbb E[U_k\mid X_t],
$$

所以它本质就是：

$$
\boxed{
V_k
=
\int_0^1
\mathbb E[
\operatorname{Var}
(U_k\mid X_t^{(k)})
]dt.
}
\tag{11.9}
$$

因此：

$$
V_k\text{ 大}
$$

说明：

> 同一位置可能有很多相互冲突的 conditional velocity。

而：

$$
V_k=0
$$

说明：

> 给定 $X_t$ 后速度唯一，linear interpolation 不存在 crossing-induced ambiguity。

---

# 11.3 定义 ODE straightness error

新 ODE 定义：

$$
\boxed{
S_{k+1}
=
\int_0^1
\mathbb E
\left[
\|
\dot Z_t^{(k+1)}
-
(Z_1^{(k+1)}-Z_0^{(k+1)})
\|^2
\right]dt.
}
\tag{11.10}
$$

这个量衡量：

$$
\boxed{
\text{同一条 ODE trajectory 上的速度是否恒定。}
}
$$

---

# 第十二部分：平方输运代价下降量为什么恰好等于 $V+S$？

这是最值得推一遍的地方。

首先：

$$
C_k
=
\mathbb E\|U_k\|^2.
\tag{12.1}
$$

由于：

$$
v_k
=
\mathbb E[U_k\mid X_t],
$$

使用条件期望的正交分解：

$$
\boxed{
\mathbb E\|U_k\|^2
=
\mathbb E\|v_k(X_t,t)\|^2
+
\mathbb E
\|U_k-v_k(X_t,t)\|^2.
}
\tag{12.2}
$$

对 $t$ 从 0 到 1 积分：

$$
C_k
=
\int_0^1
\mathbb E
\|v_k(X_t^{(k)},t)\|^2dt
+
V_k.
\tag{12.3}
$$

---

# 12.1 使用 marginal preservation

因为：

$$
X_t^{(k)}
\overset d=
Z_t^{(k+1)},
$$

所以：

$$
\int_0^1
\mathbb E
\|v_k(X_t^{(k)},t)\|^2dt
=
\int_0^1
\mathbb E
\|v_k(Z_t^{(k+1)},t)\|^2dt.
\tag{12.4}
$$

而：

$$
v_k(Z_t^{(k+1)},t)
=
\dot Z_t^{(k+1)}.
$$

因此：

$$
C_k
=
\int_0^1
\mathbb E
\|\dot Z_t^{(k+1)}\|^2dt
+
V_k.
\tag{12.5}
$$

---

# 12.2 再分解 ODE 速度

定义：

$$
D_{k+1}
=
Z_1^{(k+1)}-Z_0^{(k+1)}.
\tag{12.6}
$$

因为：

$$
D_{k+1}
=
\int_0^1
\dot Z_t^{(k+1)}dt.
\tag{12.7}
$$

展开：

$$
\|\dot Z_t\|^2
=
\|D\|^2
+
\|\dot Z_t-D\|^2
+
2D^T(\dot Z_t-D).
\tag{12.8}
$$

对时间积分。

最后一项：

$$
\int_0^1
2D^T(\dot Z_t-D)dt
$$

等于：

$$
2D^T
\left[
\int_0^1\dot Z_tdt-D
\right].
$$

由式 (12.7)：

$$
\int_0^1\dot Z_tdt=D.
$$

所以交叉项为：

$$
0.
$$

因此：

$$
\int_0^1
\|\dot Z_t\|^2dt
=
\|D\|^2
+
\int_0^1
\|\dot Z_t-D\|^2dt.
\tag{12.9}
$$

再求期望：

$$
\boxed{
\int_0^1
\mathbb E\|\dot Z_t^{(k+1)}\|^2dt
=
C_{k+1}
+
S_{k+1}.
}
\tag{12.10}
$$

---

# 12.3 代回去

式 (12.5)：

$$
C_k
=
\int_0^1
E\|\dot Z_t^{(k+1)}\|^2dt
+
V_k.
$$

使用式 (12.10)：

$$
C_k
=
C_{k+1}
+
S_{k+1}
+
V_k.
$$

因此：

$$
\boxed{
C_k-C_{k+1}
=
V_k+S_{k+1}.
}
\tag{12.11}
$$

这个式子极其重要。

它告诉我们：

$$
\boxed{
\text{一次 Rectification 降低掉的平方输运代价}
}
$$

正好可以拆成：

$$
\boxed{
\text{旧 coupling 的 velocity ambiguity}
}
$$

加上：

$$
\boxed{
\text{新 ODE 的 trajectory non-straightness}.
}
$$

这就是为什么论文从“convex cost 不增”能够进一步得到“Reflow 越来越直”的结论。原论文 Theorem 3.7 正是从这一平方代价分解和 telescoping sum 得到 $O(1/K)$ 的直线化结论。

---

# 第十三部分：为什么反复 Reflow 必须出现越来越直的轮次？

由：

$$
C_k-C_{k+1}
=
V_k+S_{k+1}
$$

对：

$$
k=0,\ldots,K
$$

求和：

$$
\sum_{k=0}^K
(C_k-C_{k+1})
=
\sum_{k=0}^K
(V_k+S_{k+1}).
\tag{13.1}
$$

左侧 telescoping：

$$
C_0-C_{K+1}
=
\sum_{k=0}^K
(V_k+S_{k+1}).
\tag{13.2}
$$

由于：

$$
C_{K+1}\geq0,
$$

所以：

$$
\sum_{k=0}^K
(V_k+S_{k+1})
\leq
C_0.
\tag{13.3}
$$

除以：

$$
K+1,
$$

得到平均值：

$$
\frac1{K+1}
\sum_{k=0}^K
(V_k+S_{k+1})
\leq
\frac{C_0}{K+1}.
\tag{13.4}
$$

最小值不超过平均值：

$$
\boxed{
\min_{0\leq k\leq K}
(V_k+S_{k+1})
\leq
\frac{C_0}{K+1}.
}
\tag{13.5}
$$

所以：

$$
\boxed{
V+S
=
O(1/K).
}
$$

这就是 Reflow 的渐近 straightening 理论。

要注意：

$$
\boxed{
\text{论文并没有证明每一次 Reflow 的 }S\text{ 都严格单调下降。}
}
$$

它证明的是随着迭代进行，总体上一定会出现越来越 straight / non-intersecting 的轮次，并给出 $O(1/K)$ 控制。原论文也提醒实际中不建议无限 Reflow，因为模型近似误差可能累积。

---

# 第十四部分：为什么 Reflow 最终有可能做到一步生成？

理想情况下：

$$
V_k\rightarrow0,
$$

意味着：

$$
U_k
=
\mathbb E[U_k\mid X_t].
\tag{14.1}
$$

因此训练 target 与 marginal velocity 不再存在冲突。

同时：

$$
S_{k+1}\rightarrow0,
$$

意味着：

$$
\dot Z_t
=
Z_1-Z_0
=
\mathrm{const}.
\tag{14.2}
$$

因此：

$$
Z_t
=
Z_0+t(Z_1-Z_0).
\tag{14.3}
$$

于是：

$$
v(Z_t,t)
=
Z_1-Z_0.
$$

特别在：

$$
t=0
$$

时：

$$
v(Z_0,0)
=
Z_1-Z_0.
$$

所以：

$$
\boxed{
Z_1
=
Z_0+v(Z_0,0).
}
\tag{14.4}
$$

一个 Euler step 就能精确到终点。

这就是：

$$
\boxed{
\text{straight flow}
\Longrightarrow
\text{one-step generation}.
}
$$

---

# 第十五部分：Straight coupling 和 Optimal Transport 不是一回事

这里非常容易误解。

Rectification 保证：

$$
C_{k+1}(c)
\leq
C_k(c)
$$

对所有凸 $c$ 成立。

但这不意味着：

$$
\gamma_k
\rightarrow
\gamma_{\mathrm{OT}}
$$

一定成立。

论文明确指出，高维情况下：

$$
\boxed{
\text{straight coupling}
\neq
\text{某个特定 cost 的 optimal coupling}
}
$$

一般成立。

如果某 coupling 对严格凸 cost 已经全局最优，那么它一定是 straight；但反过来不一定。

也就是说：

$$
\boxed{
\text{optimal}
\Longrightarrow
\text{straight},
}
$$

但高维中：

$$
\boxed{
\text{straight}
\not\Longrightarrow
\text{optimal}.
}
$$

一维比较特殊。

一维中，非交叉 coupling 就对应 monotone coupling；在适当条件下它正好也是凸输运代价的最优 coupling。

---

# 第十六部分：为什么线性 interpolation 如此重要？

论文也考虑一般路径：

$$
X_t
=
\alpha_tX_1+\beta_tX_0.
\tag{16.1}
$$

此时：

$$
\dot X_t
=
\dot\alpha_tX_1
+
\dot\beta_tX_0.
\tag{16.2}
$$

仍然可以训练：

$$
\boxed{
\min_v
\int_0^1
E\|
v(X_t,t)-\dot X_t
\|^2dt.
}
\tag{16.3}
$$

MSE 最优仍然是：

$$
v^*(x,t)
=
E[\dot X_t\mid X_t=x].
\tag{16.4}
$$

因此 **marginal preserving property 仍然成立**。

但如果路径不是 canonical linear path：

$$
X_t=(1-t)X_0+tX_1,
$$

那么论文的两个关键性质一般不再自动成立：

$$
\boxed{
\text{convex transport cost 不增}
}
$$

以及：

$$
\boxed{
\text{Reflow straightening}.
}
$$

原论文专门指出一般 nonlinear interpolation 仍能保持 marginals，但不再保证 convex-cost decrease 或 straightening。

所以 Rectified Flow 不只是“随便一种 Flow Matching”。

它的特殊理论性质和：

$$
\boxed{
\text{匀速线性 interpolation}
}
$$

密切相关。

---

# 第十七部分：Rectified Flow 和普通 Flow Matching 的关系

如果 Flow Matching 同样采用：

$$
X_t=(1-t)X_0+tX_1,
$$

以及：

$$
u_t=X_1-X_0,
$$

那么训练目标：

$$
\mathbb E
\|
v_\theta(X_t,t)
-
(X_1-X_0)
\|^2
\tag{17.1}
$$

与基础 Rectified Flow 完全相同。

区别主要在研究重点。

Flow Matching 更强调：

$$
\boxed{
\text{如何构造 probability path 和 conditional vector field，
并通过 CFM 学 marginal vector field。}
}
$$

Rectified Flow 更强调：

$$
\boxed{
\text{这个 ODE 如何改变 coupling；
为什么 convex transport cost 不增；
为什么 Reflow 可以把 flow 变直。}
}
$$

因此从训练代码看，两者可能几乎一样；

从理论问题看，Rectified Flow 在研究：

$$
\boxed{
\gamma_0
\rightarrow
\gamma_1
\rightarrow
\gamma_2
\rightarrow\cdots.
}
$$

---

# 第十八部分：整篇论文最重要的逻辑链

现在把所有东西串起来。

初始：

$$
\boxed{
(X_0,X_1)\sim\gamma_0.
}
\tag{18.1}
$$

构造 conditional straight paths：

$$
\boxed{
X_t=(1-t)X_0+tX_1.
}
\tag{18.2}
$$

每条路径 conditional velocity：

$$
\boxed{
U=X_1-X_0.
}
\tag{18.3}
$$

但同一个 $X_t$ 可能对应很多 $U$。

因此网络通过 MSE 学：

$$
\boxed{
v(x,t)
=
E[U\mid X_t=x].
}
\tag{18.4}
$$

这个边际速度驱动：

$$
\boxed{
\dot Z_t=v(Z_t,t).
}
\tag{18.5}
$$

虽然：

$$
Z_t\neq X_t
$$

对单个粒子一般成立，但：

$$
\boxed{
\operatorname{Law}(Z_t)
=
\operatorname{Law}(X_t).
}
\tag{18.6}
$$

因此 ODE 的 endpoints 形成新的合法 coupling：

$$
\boxed{
\gamma_1
=
\operatorname{Law}(Z_0,Z_1).
}
\tag{18.7}
$$

两次 Jensen 给出：

$$
\boxed{
C(\gamma_1)
\leq
C(\gamma_0)
}
\tag{18.8}
$$

对所有凸位移 cost 成立。

平方代价下进一步有：

$$
\boxed{
C_k-C_{k+1}
=
V_k+S_{k+1}.
}
\tag{18.9}
$$

其中：

$$
V_k
=
\text{同一 }(x,t)\text{ 上 conditional velocities 的冲突},
$$

而：

$$
S_{k+1}
=
\text{同一 ODE trajectory 上速度随时间的变化}.
$$

Reflow：

$$
\boxed{
\gamma_0
\rightarrow
\gamma_1
\rightarrow
\gamma_2
\rightarrow\cdots
}
\tag{18.10}
$$

不断重新组织 coupling。

由于：

$$
\boxed{
\min_{k\leq K}
(V_k+S_{k+1})
=
O(1/K),
}
\tag{18.11}
$$

最终得到越来越接近：

$$
\boxed{
v(Z_t,t)=Z_1-Z_0=\mathrm{const}
}
$$

的 flow。

于是：

$$
\boxed{
Z_t=(1-t)Z_0+tZ_1,
}
$$

最后可以用非常少的积分步甚至一步生成。

---

# 第十九部分：最值得记住的五句话

第一，**coupling 不只是两个边缘分布**。它描述的是：

$$
\boxed{
X_0\text{ 和 }X_1\text{ 怎么配对。}
}
$$

第二，训练 target：

$$
U=X_1-X_0
$$

是每个 endpoint pair 的 conditional velocity，而模型真正学习的是：

$$
\boxed{
v(x,t)=E[U\mid X_t=x].
}
$$

第三，$X_t$ 和真正生成轨迹 $Z_t$ 不是同一条路径：

$$
\boxed{
X_t\neq Z_t,
\qquad
X_t\overset d=Z_t.
}
$$

第四，凸输运代价不增来自两次 Jensen：

$$
\boxed{
U
\rightarrow E[U\mid X_t]
\rightarrow
\int_0^1vdt.
}
$$

第一层消除 velocity ambiguity，第二层把时间变化的速度压成最终净位移。

第五，Reflow 的本质不是简单“重复训练”，而是：

$$
\boxed{
\text{迭代优化 coupling。}
}
$$

每次：

$$
\gamma_k
\rightarrow
v_k
\rightarrow
\gamma_{k+1},
$$

再把新 endpoints 重新拉成直线训练。

这就是 Rectified Flow 从任意随机 pairing，逐渐得到低冲突、低曲率、适合少步生成的确定性 transport 的完整逻辑。